# 04 · V-JEPA 2.1-B + dense CAN head smoke test

This notebook verifies model loading, shapes, GPU memory, forward pass and one backward pass before any long training run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
DATA_ROOT = DRIVE_ROOT / 'DATASET'
COMMA_ROOT = DATA_ROOT / 'comma2k19'
RAW_ROOT = COMMA_ROOT / 'raw'
PROCESSED_ROOT = COMMA_ROOT / 'processed' / 'v1'
MANIFEST_ROOT = DRIVE_ROOT / 'manifests' / 'stage3' / 'v1'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'stage3'
PRETRAINED_ROOT = DRIVE_ROOT / 'pretrained'

# Clone your repository if this runtime does not have it yet.
if not REPO.exists():
    raise RuntimeError('Clone Blackbox-Detection to /content/Blackbox-Detection first, then rerun this cell.')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT      :', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
# Install this repository through its existing pyproject.toml without replacing
# Colab's binary stack. Dependency versions in pyproject.toml are aligned to the
# DACON evaluation-server package list.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)


In [ ]:
VJEPA_REPO = Path('/content/vjepa2')
VJEPA_COMMIT = '45d025f636dfc58fc2426905fc4a1ab755b1c3e5'  # V-JEPA 2.1 release commit; pin for reproducibility
if not VJEPA_REPO.exists():
    !git clone -q https://github.com/facebookresearch/vjepa2.git /content/vjepa2
!git -C /content/vjepa2 checkout -q $VJEPA_COMMIT
print('V-JEPA commit:', subprocess.check_output(['git','-C',str(VJEPA_REPO),'rev-parse','HEAD'], text=True).strip())

VJEPA_CKPT = PRETRAINED_ROOT / 'vjepa2_1_vitb_dist_vitG_384.pt'
if not VJEPA_CKPT.exists():
    !wget -q -O "$VJEPA_CKPT" https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt
print(VJEPA_CKPT, VJEPA_CKPT.stat().st_size/2**20, 'MiB')

In [ ]:
import json, torch
from torch.utils.data import DataLoader
from blackbox_detection.stage3.dataset import Stage3CANDataset
from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN
from blackbox_detection.stage3.losses import can_multitask_loss

stats_path = MANIFEST_ROOT / 'target_stats.json'
train_manifest = MANIFEST_ROOT / 'comma_train.csv'
stats = json.loads(stats_path.read_text())

ds = Stage3CANDataset(train_manifest, PROCESSED_ROOT, target_stats=stats, clip_len=16, window_stride=16, input_size=(288,384), random_flip=True, max_windows=16)
loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
print('video:', batch['video'].shape, batch['video'].dtype)
print('target:', batch['target'].shape)

In [ ]:
device = torch.device('cuda')
backbone = load_vjepa21_base_encoder(VJEPA_REPO, VJEPA_CKPT, num_frames=16, out_layers=(2,5,8,11), freeze=True)
model = VJEPA21DenseCAN(backbone, freeze_backbone=True).to(device)

video = batch['video'].to(device)
target = batch['target'].to(device)
valid = batch['valid'].to(device)
with torch.autocast('cuda', dtype=torch.bfloat16):
    out = model(video)
    loss, parts = can_multitask_loss(out, target, valid)
print({k:v.shape for k,v in out.items()})
print('loss:', loss.item(), parts)
loss.backward()
print('max allocated GiB:', torch.cuda.max_memory_allocated()/2**30)

If this cell fits comfortably on the L4, notebook 05 can use batch size 2. If memory is tight, use batch size 1 and increase gradient accumulation.